# 03 - Foundry IQ: grounding the agent in policy documents

**Maple Mortgage Assistant, Northwind Home Lending (fictional)**

## What you'll learn
- Chunk and index policy documents in **Azure AI Search** (vector + keyword + semantic ranker)
- Baseline retrieval: a plain hybrid query
- **Foundry IQ**: a *knowledge source* + *knowledge base* that plans queries, retrieves and returns citations
- Ground a Foundry prompt agent through the knowledge base MCP endpoint, show **citations**, and see it say "I don't know" when the corpus has no answer
- When to use classic RAG vs agentic retrieval

> **[preview]** Knowledge bases / knowledge sources and the KB MCP endpoint are preview (`2026-08-01-preview`). Cells that need them print the real error if your region/tenant lacks the capability; nothing is simulated.

In [1]:
import os, sys, asyncio, json
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

DAY2 = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(DAY2 / ".env", override=False)
sys.path.insert(0, str(DAY2))

REQUIRED = ["AZURE_AI_PROJECT_ENDPOINT", "AZURE_AI_MODEL_DEPLOYMENT_NAME", "AZURE_AI_EMBEDDING_DEPLOYMENT_NAME", "AZURE_SEARCH_ENDPOINT", "AZURE_SEARCH_INDEX_NAME", "AZURE_SEARCH_KNOWLEDGE_BASE_NAME", "AZURE_AI_KB_CONNECTION_NAME"]
for name in REQUIRED:
    print(f"{name:42s}", "set" if os.environ.get(name) else "MISSING")   # never print values
assert all(os.environ.get(n) for n in REQUIRED), "Run scripts/write-env.sh first"

credential = DefaultAzureCredential()

import re, time
from urllib.parse import urlparse
from azure.ai.projects import AIProjectClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient

ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
CHAT, EMBED = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"], os.environ["AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"]
SEARCH = os.environ["AZURE_SEARCH_ENDPOINT"]
INDEX, KB = os.environ["AZURE_SEARCH_INDEX_NAME"], os.environ["AZURE_SEARCH_KNOWLEDGE_BASE_NAME"]
KB_CONN = os.environ["AZURE_AI_KB_CONNECTION_NAME"]
KS = f"{INDEX}-source"                                     # knowledge source name
OPENAI_URL = f"https://{urlparse(ENDPOINT).hostname.split('.')[0]}.openai.azure.com"   # Azure OpenAI endpoint of the Foundry account

pc = AIProjectClient(endpoint=ENDPOINT, credential=credential, allow_preview=True)
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI
aoai = AzureOpenAI(azure_endpoint=OPENAI_URL, api_version="2024-10-21",
                   azure_ad_token_provider=get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default"))
oai = pc.get_openai_client()      # project client: used for agents (Responses API)
index_client = SearchIndexClient(SEARCH, credential)
print("search index:", INDEX, "| knowledge base:", KB)

AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set
AZURE_SEARCH_ENDPOINT                      set
AZURE_SEARCH_INDEX_NAME                    set
AZURE_SEARCH_KNOWLEDGE_BASE_NAME           set
AZURE_AI_KB_CONNECTION_NAME                set


search index: northwind-policies | knowledge base: northwind-kb


## 1. Chunk the policy documents
One chunk per `##` section, each prefixed with the policy id and title so the chunk stands alone. Small, self-contained chunks give precise citations.

In [2]:
POLICY_DIR = DAY2 / "data" / "policies"
chunks = []
for f in sorted(POLICY_DIR.glob("*.md")):
    text = f.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()
    policy_id = title.split()[0]
    for n, part in enumerate(re.split(r"\n(?=## )", text)[1:], start=1):
        heading = part.splitlines()[0].lstrip("# ").strip()
        chunks.append({"id": f"{f.stem}-{n}", "policy_id": policy_id, "title": title, "section": heading,
                       "source": f.name, "content": f"{title} / {heading}\n" + "\n".join(part.splitlines()[1:]).strip()})
print(len(chunks), "chunks from", len(list(POLICY_DIR.glob('*.md'))), "files")
print(chunks[3]["id"], "|", chunks[3]["section"], "|", len(chunks[3]["content"]), "chars")

40 chunks from 8 files
01-eligibility-4 | 4. Credit score bands | 347 chars


## 2. Create the index
Vector field (HNSW, 1536 dims for `text-embedding-3-small`), a vectorizer so queries embed themselves (the Search service's managed identity calls the embedding deployment; no keys), and a semantic configuration.

In [3]:
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType, VectorSearch,
    HnswAlgorithmConfiguration, VectorSearchProfile, AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch)

index = SearchIndex(
    name=INDEX,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SimpleField(name="policy_id", type=SearchFieldDataType.String, filterable=True, facetable=True),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchableField(name="section", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String, filterable=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SearchField(name="content_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                    searchable=True, vector_search_dimensions=1536, vector_search_profile_name="hnsw-profile"),
    ],
    vector_search=VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
        profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw", vectorizer_name="aoai")],
        vectorizers=[AzureOpenAIVectorizer(vectorizer_name="aoai", parameters=AzureOpenAIVectorizerParameters(
            resource_url=OPENAI_URL, deployment_name=EMBED, model_name="text-embedding-3-small"))],
    ),
    semantic_search=SemanticSearch(default_configuration_name="semantic", configurations=[SemanticConfiguration(
        name="semantic", prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"), content_fields=[SemanticField(field_name="content")]))]),
)
index_client.create_or_update_index(index)
print("index ready:", INDEX)

index ready: northwind-policies


## 3. Embed and upload
Embeddings come from the project's OpenAI-compatible client (Entra auth).

In [4]:
vectors = []
for i in range(0, len(chunks), 16):
    batch = [c["content"] for c in chunks[i:i + 16]]
    vectors += [d.embedding for d in aoai.embeddings.create(model=EMBED, input=batch).data]
docs = [{**c, "content_vector": v} for c, v in zip(chunks, vectors)]

search = SearchClient(SEARCH, INDEX, credential)
result = search.upload_documents(docs)
print("uploaded:", sum(r.succeeded for r in result), "of", len(docs))
for _ in range(20):                      # indexing is near-real-time
    if search.get_document_count() == len(docs):
        break
    time.sleep(2)
print("documents in index:", search.get_document_count())

uploaded: 40 of 40


documents in index: 40


## 4. Baseline: hybrid + semantic query
This is classic RAG retrieval: you write the query, you format the results.

In [5]:
from azure.search.documents.models import VectorizableTextQuery
q = "What is the minimum down payment on a 600k home?"
hits = search.search(search_text=q, vector_queries=[VectorizableTextQuery(text=q, k_nearest_neighbors=5, fields="content_vector")],
                     query_type="semantic", semantic_configuration_name="semantic", top=3,
                     select=["id", "policy_id", "section", "content"])
for h in hits:
    print(f"{h['@search.reranker_score']:.2f}  {h['policy_id']} / {h['section']}")

2.37  NHL-POL-101 / 3. Down payment
2.08  NHL-POL-101 / 2. Property types
1.56  NHL-POL-103 / 1. All applicants


## 5. Foundry IQ: knowledge source + knowledge base [preview]
- **Knowledge source** = pointer to the index (which fields to return, which semantic config).
- **Knowledge base** = the reasoning layer: an LLM plans sub-queries from the chat, runs them against the sources, and returns grounded results with references.

In [6]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters, KnowledgeBase,
    KnowledgeSourceReference, KnowledgeBaseAzureOpenAIModel, AzureOpenAIVectorizerParameters, SearchIndexFieldReference)

ks = SearchIndexKnowledgeSource(
    name=KS, description="Northwind underwriting and compliance policies",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX, semantic_configuration_name="semantic",
        source_data_fields=[SearchIndexFieldReference(name=n) for n in ("id", "policy_id", "section", "source", "content")]))
index_client.create_or_update_knowledge_source(ks)

kb = KnowledgeBase(
    name=KB, description="Maple policy knowledge base",
    knowledge_sources=[KnowledgeSourceReference(name=KS)],
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
        resource_url=OPENAI_URL, deployment_name=CHAT, model_name=CHAT))],
    retrieval_instructions="Northwind mortgage policies. Cite the policy id and section.",
    answer_instructions="Answer only from the retrieved passages. If they do not contain the answer say you do not know.",
)
index_client.create_or_update_knowledge_base(kb)
print("knowledge source:", KS, "| knowledge base:", KB)

knowledge source: northwind-policies-source | knowledge base: northwind-kb


### Query the knowledge base directly
`KnowledgeBaseRetrievalClient.retrieve` returns passages plus references (what the MCP tool uses under the hood).

In [7]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import KnowledgeBaseRetrievalRequest, KnowledgeRetrievalSemanticIntent

kb_client = KnowledgeBaseRetrievalClient(SEARCH, credential, knowledge_base_name=KB)
resp = kb_client.retrieve(KnowledgeBaseRetrievalRequest(
    intents=[KnowledgeRetrievalSemanticIntent(search="maximum total debt service ratio and gift funds down payment")],
    include_activity=True))
print("references:", len(resp.references or []))
for r in resp.references or []:
    d = r.as_dict()
    print(f" - {d.get('docKey'):18s} {d.get('title')}  (reranker {float(d.get('rerankerScore') or 0):.2f})")
print("activity steps:", [a.as_dict().get("type") for a in resp.activity or []])

references: 5
 - 02-dti-limits-2    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.42)
 - 02-dti-limits-1    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.41)
 - 02-dti-limits-3    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.30)
 - 02-dti-limits-4    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 2.27)
 - 02-dti-limits-5    NHL-POL-102 Debt Service Ratio (DTI) Limits  (reranker 1.94)
activity steps: ['searchIndex', 'agenticReasoning']


## 6. Ground a Foundry agent through the KB MCP endpoint
The project connection `northwind-kb-mcp` (created by the Bicep deployment, `RemoteTool` with the project's managed identity) points at
`<search>/knowledgebases/<kb>/mcp`. The agent gets one tool, `knowledge_base_retrieve`.

In [8]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

AGENT = "maple-policy-iq"
mcp_url = f"{SEARCH}/knowledgebases/{KB}/mcp?api-version=2026-08-01-preview"
instructions = (
    "You are Maple, the underwriting-policy assistant for Northwind Home Lending (fictional). "
    "ALWAYS call knowledge_base_retrieve before answering a policy question. Answer only from the retrieved passages, "
    "and cite them as 【policy id, section】. If the passages do not answer the question, reply exactly: "
    "\"I don't know - escalating to a human underwriter.\" Never promise a rate or approval.")
agent = pc.agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(model=CHAT, instructions=instructions, tools=[
        MCPTool(server_label="knowledge-base", server_url=mcp_url, require_approval="never",
                allowed_tools=["knowledge_base_retrieve"], project_connection_id=KB_CONN)]))
print("agent:", agent.name, "version", agent.version)

agent: maple-policy-iq version 1


In [9]:
def ask(question, tries=5):
    """Call the agent; back off on 429 / transient tool errors (shared, low-quota sandbox deployment)."""
    for attempt in range(tries):
        try:
            r = oai.responses.create(input=question, extra_body={"agent_reference": {"name": AGENT, "type": "agent_reference"}})
            break
        except Exception as exc:
            if attempt == tries - 1:
                raise
            wait = 15 * (attempt + 1)
            print(f"  retry in {wait}s ({type(exc).__name__}: {str(exc)[:80]})")
            time.sleep(wait)
    tools = [i for i in r.output if i.type in ("mcp_call", "mcp_list_tools")]
    print("Q:", question)
    print("tool calls:", [(i.type, getattr(i, "name", None), getattr(i, "error", None)) for i in tools])
    print("A:", r.output_text, "\n")
    return r

r1 = ask("What documents does a self-employed applicant need, and can they be pre-qualified automatically?")
r2 = ask("What is the minimum down payment on a $600,000 home?")

Q: What documents does a self-employed applicant need, and can they be pre-qualified automatically?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: A self-employed applicant needs to provide the following documents:
- Two years of Notice of Assessment and T1 General.
- Two years of financial statements (for incorporated businesses) and business registration.

Additionally, self-employed applicants are always referred for manual underwriting and cannot be pre-qualified automatically. The policy mandates a mandatory escalation to a human underwriter for self-employed applicants, which means pre-qualification cannot be automatic in their case.

This is supported by the policy that states self-employed applicants require two years of Notice of Assessment history and are always referred for manual review. The escalation policy also states that the assistant must escalate all self-employed applicants to a human underwriter.

Thus, automatic pre-

Q: What is the minimum down payment on a $600,000 home?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: The minimum down payment on a $600,000 home is calculated as follows:

- 5% on the first $500,000, which is $25,000
- 10% on the remaining $100,000, which is $10,000

So, the total minimum down payment would be $25,000 + $10,000 = $35,000.

This means the minimum down payment on a $600,000 home is $35,000. Note that down payments under 20% require mortgage default insurance【4:0†NHL-POL-101 Mortgage Eligibility Policy, section 3. Down payment】. 



### Out of scope: it should say it does not know

In [10]:
r3 = ask("What is Northwind's policy on cryptocurrency income and NFT collateral?")
r3b = ask("Can gift funds be used for the down payment?")   # not covered by the corpus either

Q: What is Northwind's policy on cryptocurrency income and NFT collateral?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: I don't know - escalating to a human underwriter. 



Q: Can gift funds be used for the down payment?
tool calls: [('mcp_list_tools', None, None), ('mcp_call', 'knowledge_base_retrieve', None)]
A: The retrieved policy excerpts do not explicitly address whether gift funds can be used for the down payment. They only specify down payment amounts required and documentation around proof of down payment, but no mention of gift funds is made.

I don't know - escalating to a human underwriter. 



### Show the citations
The retrieved passages are in the `mcp_call` output; the references name the policy documents.

In [11]:
def show_citations(resp):
    """Print policy id + section of every passage the KB returned for this response."""
    for item in resp.output:
        if item.type == "mcp_call" and getattr(item, "output", None):
            for d in json.loads(item.output).get("documents", []):
                c = json.loads(d["content"])
                print(f"  [{c['policy_id']} / {c['section']}]  ({c['source']})")
show_citations(r2)
for item in r2.output:
    if item.type == "message":
        for part in item.content:
            for a in getattr(part, "annotations", []) or []:
                print("annotation:", a.type, "->", str(getattr(a, "url", "")).split("/docs/")[-1].split("?")[0])

  [NHL-POL-101 / 3. Down payment]  (01-eligibility.md)
  [NHL-POL-101 / 2. Property types]  (01-eligibility.md)
  [NHL-POL-103 / 1. All applicants]  (03-document-checklist.md)
  [NHL-POL-101 / 4. Credit score bands]  (01-eligibility.md)
  [NHL-POL-102 / 1. Definitions]  (02-dti-limits.md)
  [NHL-POL-101 / 1. Who can apply]  (01-eligibility.md)
annotation: url_citation -> 01-eligibility-3


## 7. Alternative: Agent Framework `AzureAISearchContextProvider` [preview package]
Same knowledge base, but retrieval runs in your process and the agent is code-defined. Use it when you build with Agent Framework rather than server-side Foundry agents.

In [12]:
from agent_framework import Agent
from agent_framework.azure import AzureAISearchContextProvider
from agent_framework.foundry import FoundryChatClient

provider = AzureAISearchContextProvider(
    endpoint=SEARCH, credential=credential, mode="agentic", knowledge_base_name=KB,
    azure_openai_resource_url=OPENAI_URL, model=CHAT)
af_agent = Agent(client=FoundryChatClient(project_endpoint=ENDPOINT, model=CHAT, credential=credential),
                 name="maple-af-iq", instructions="Answer only from the provided context and name the policy id. If unsure say you do not know.",
                 context_providers=[provider])
res = await af_agent.run("What is the maximum total debt service ratio?")
print(res.text)

The maximum total debt service (TDS) ratio is 44%. The policy id is NHL-POL-102.


## Key takeaways
- Index = your data; knowledge source = a pointer; knowledge base = planning + retrieval + references, reusable by many agents.
- Classic RAG (section 4): you control the query, cheap and predictable. Agentic retrieval (sections 5-7): better for multi-part, conversational questions; costs extra LLM calls.
- Chunk by section and keep the policy id in the chunk: citations become auditable.
- Instruct the agent to refuse when retrieval is empty. For FSI, an "I don't know, escalating" path beats a confident guess.
- Auth is Entra / managed identity throughout; no keys. Search needs `Search Index Data Reader` for the project identity.

**Next up:** [04 - Memory](04-memory.ipynb)